# Task 02 — Silver: Orders Clean & Enrich

**Workshop: Final Pipeline | Layer 2 of 3**

## Goal

Read `bronze.lab_orders`, recast types and apply transformations, write to `silver.lab_orders`.

```
bronze.lab_orders  (all source columns = StringType)
        |
        v  recast * withColumn * when/otherwise * filter * drop
        v
silver.lab_orders  (typed, enriched, clean)
```

## Transformations to implement

| # | Exercise | What | Details |
|---|----------|------|---------|
| 1 | Ex 3 | `cast` | `order_datetime` → TimestampType; `total_amount`, `discount_percent` → DoubleType; `quantity` → IntegerType |
| 2 | Ex 4 | `withColumn` | Add `net_amount` = `total_amount` × (1 - `discount_percent` / 100), rounded to 2 dp |
| 3 | Ex 4 | `when / otherwise` | Add `order_tier`: `"Premium"` ≥ 500 · `"Standard"` ≥ 200 · `"Small"` otherwise |
| 4 | Ex 4 | `filter` | Remove rows where `order_id` IS NULL |
| 5 | Ex 4 | `drop` | Remove `_source_file` (Bronze metadata, not needed in Silver) |

> `net_amount` is used by Task 03 for revenue — make sure the formula and types are correct.


In [ ]:
%run ../../../setup/00_setup

In [ ]:
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("bronze_schema", BRONZE_SCHEMA, "Bronze Schema")
dbutils.widgets.text("silver_schema", SILVER_SCHEMA, "Silver Schema")

catalog       = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

source_table = f"{catalog}.{bronze_schema}.lab_orders"
target_table = f"{catalog}.{silver_schema}.lab_orders"

print(f"Source : {source_table}")
print(f"Target : {target_table}")

---

## Exercise 1 — Imports

Import all PySpark functions needed for the five transformations above.


**💡 Guidance — Exercise 1**

Import the transformation functions needed for the five Silver operations.

**Key functions to import**
- `col` — reference a DataFrame column by name in expressions
- `to_timestamp(col_expr, format)` — parse a string column into `TimestampType` using a Java format pattern. For this dataset use `"yyyy-MM-dd'T'HH:mm:ss"`. The single quotes around `T` escape the literal character in the pattern.
- `when(cond, val).when(...).otherwise(val)` — conditional expression (SQL `CASE WHEN`); always end the chain with `.otherwise()` to cover all rows
- `round(col_expr, scale)` — rounds a `Column` expression to N decimal places. The first argument **must** be a `Column`, not a Python float.
- `lit(value)` — wraps a Python scalar as a constant `Column` expression. Required when mixing Python numbers with column arithmetic: `col("a") * (lit(1) - col("b") / lit(100))`

**Important note on shadowing**
Importing `round` and `sum` from PySpark shadows Python's built-ins of the same names. This is intentional — the PySpark versions operate on `Column` expressions, not Python scalars.

**Things to think about**
- What is the difference between `round(2.345, 2)` (Python built-in) and `round(col("amount"), 2)` (PySpark `Column`)?
- Why do we need `lit(1)` instead of just `1` when writing `col("total") * (lit(1) - col("discount") / lit(100))`?

In [ ]:
# YOUR CODE HERE

---

## Exercise 2 — Read the Bronze table


**💡 Guidance — Exercise 2**

Read the Bronze `lab_orders` table into a DataFrame before applying transformations.

**Reading with spark.table()**
`spark.table("catalog.schema.table_name")` is the cleanest way to read a managed Delta table from Unity Catalog. It is equivalent to `spark.sql("SELECT * FROM ...")` but more concise in a Python chain. No file path is needed — Unity Catalog manages the location.

The Bronze table was written by Task 01 Notebook — that notebook must have run successfully before you can execute this cell. Store the result as `orders_bronze`.

**Expected state of the Bronze data**
At this point `orders_bronze` should contain the 11 original JSON columns plus `_load_ts` and `_source_file`. The `order_datetime` column is still a raw string — you will cast it in Exercise 3.

**Things to think about**
- When would you use `spark.read.format("delta").load("/path")` instead of `spark.table("name")`?
- What happens to the Silver transformation chain if the Bronze table is empty?

In [ ]:
# Read the Bronze table into orders_bronze (Task 01 must have run first)
orders_bronze = # YOUR CODE HERE

In [ ]:
print(f"Bronze rows : {orders_bronze.count():,}")
orders_bronze.printSchema()

---

## Exercise 3 — Inspect schema and recast columns

Look at the Bronze schema printed above. Bronze stores **all** source columns as `StringType` — no type coercion at ingestion. Cast the following columns to their correct types before building the Silver layer:

| Column | Bronze type | Silver type |
|--------|-------------|-------------|
| `order_datetime` | `StringType` | `TimestampType` |
| `total_amount` | `StringType` | `DoubleType` |
| `discount_percent` | `StringType` | `DoubleType` |
| `quantity` | `StringType` | `IntegerType` |

Store the result back as `orders_bronze`.


**💡 Guidance — Exercise 3**

Bronze stores every source column as `StringType` — this is intentional (Bronze = raw, no interpretation). Silver is responsible for coercing columns into their proper types.

**Four casts to apply**

Use `.withColumn("col_name", col("col_name").cast(TargetType()))` to overwrite a column in-place:

| Column | Target type | Why |
|--------|-------------|-----|
| `order_datetime` | `TimestampType` | Enables date-based groupBy and window functions |
| `total_amount` | `DoubleType` | Needed for net_amount arithmetic in Exercise 4 |
| `discount_percent` | `DoubleType` | Needed for net_amount formula |
| `quantity` | `IntegerType` | Needed for any numeric aggregation on qty |

Chain all four `.withColumn()` calls in a single expression.  
Alternatively, use `to_timestamp(col("order_datetime"), "yyyy-MM-dd'T'HH:mm:ss")` for the datetime column — it is equivalent to `.cast(TimestampType())` but handles the format pattern explicitly.

**Things to think about**
- What happens to rows where `order_datetime` doesn't match the format? (hint: they become `null`)
- What would happen in Exercise 4 if you forgot to cast `total_amount` before doing arithmetic on it?


In [ ]:
# Inspect schema — all source columns are StringType from Bronze
orders_bronze.printSchema()

# Recast four columns to their correct types and store back as orders_bronze:
#   order_datetime  -> TimestampType  (use to_timestamp with format pattern)
#   total_amount    -> DoubleType
#   discount_percent-> DoubleType
#   quantity        -> IntegerType
orders_bronze = (
    orders_bronze
    # YOUR CODE HERE
)


In [ ]:
# Verify — check that all four columns show the correct types
orders_bronze.select(
    "order_id", "order_datetime", "total_amount", "discount_percent", "quantity"
).show(5, truncate=False)
orders_bronze.printSchema()


---

## Exercise 4 — Apply the remaining four transformations in one chain

`order_datetime` is already a `TimestampType` (Exercise 3). Apply the remaining four transformations in a single chained expression. Name the result `orders_silver`.


**💡 Guidance — Exercise 4**

After Exercise 3, `order_datetime` is `TimestampType` and `total_amount` / `discount_percent` are `DoubleType` — arithmetic will work correctly. Apply the four remaining transformations in a single chain.

**Transformation 1 — Net amount**
`total_amount × (1 − discount_percent / 100)`. Both columns are now DoubleType, but use `lit()` to wrap Python integer constants: `round(col("total_amount") * (lit(1) - col("discount_percent") / lit(100)), 2)`. Name the column `net_amount`.

**Transformation 2 — Order tier**
Use `when().when().otherwise()` on `net_amount`: `≥ 500 → "Premium"`, `≥ 200 → "Standard"`, otherwise `"Small"`. Place the most restrictive condition first. `net_amount` must be added **before** this step in the chain.

**Transformation 3 — Filter nulls**
Remove rows where `order_id` is null: `.filter(col("order_id").isNotNull())`.

**Transformation 4 — Drop metadata column**
Remove `_source_file` with `.drop("_source_file")`.

**Things to think about**
- Does the order of these transformations matter? What would break if `order_tier` was computed before `net_amount` was added?
- Why is `round(avg(...), 2)` nested — what type does `avg(...)` return?


In [ ]:
# Apply the 4 remaining transformations in a single chain:
#  1. add net_amount = round(total_amount * (1 - discount_percent / 100), 2)
#  2. add order_tier: "Premium" >= 500 | "Standard" >= 200 | "Small" otherwise
#  3. filter rows where order_id IS NOT NULL
#  4. drop _source_file
orders_silver = (
    orders_bronze
    # YOUR CODE HERE
)


In [ ]:
print(f"Silver rows : {orders_silver.count():,}")
orders_silver.select(
    "order_id","order_datetime","total_amount",
    "discount_percent","net_amount","order_tier"
).show(10, truncate=False)
orders_silver.printSchema()

---

## Exercise 5 — Write to Silver as a managed Delta table


**💡 Guidance — Exercise 5**

Write `orders_silver` to a managed Delta table as the Silver layer of the pipeline.

**Write options**
Use `mode("overwrite")` — Silver is always fully rebuilt from Bronze on each pipeline run. Set `option("overwriteSchema", "true")` because the Silver schema contains new columns (`net_amount`, `order_tier`) that were not present in Bronze. Without this option, Spark raises an `AnalysisException` if the target table already exists with a different schema from a previous run.

**Full refresh pattern**
Rebuilding Silver entirely on each run ensures that any fix applied to the Bronze data propagates through automatically — no incremental state to manage.

**Things to think about**
- Why is Silver always rebuilt from Bronze rather than updated incrementally?
- What would happen if you saved `orders_silver` before applying the null filter in Exercise 4?


In [ ]:
# Write orders_silver to target_table as a managed Delta table
# mode: overwrite | overwriteSchema: true (Silver adds new columns not in Bronze)
# YOUR CODE HERE

In [ ]:
row_count = spark.table(target_table).count()
print(f"OK  {target_table}  ->  {row_count:,} rows")
display(spark.table(target_table).limit(5))

In [ ]:
import json
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))